In [35]:
import json
from pathlib import Path
import pandas as pd
from rouge_score import rouge_scorer

In [36]:
from pathlib import Path

BASE_DIR = Path.cwd().parent  

SUMMARY_PATH = BASE_DIR / "data" / "summaries" / "akzhan_sample4_summary.json"
GOLD_PATH = BASE_DIR / "data" / "gold_summaries" / "sample_4_gold.json"


In [37]:
with open(SUMMARY_PATH, "r", encoding="utf-8") as f:
    system_summary = json.load(f)

with open(GOLD_PATH, "r", encoding="utf-8") as f:
    gold = json.load(f)

reference = gold["human_summary"] if isinstance(gold, dict) else gold

extractive = system_summary.get("textrank_summary", "")
abstractive = system_summary.get("abstractive_full_summary", "")

In [38]:
scorer = rouge_scorer.RougeScorer([
    "rouge1",
    "rouge2",
    "rougeL"
], use_stemmer=True)

In [39]:
scores_extractive = scorer.score(reference, extractive)
scores_abstractive = scorer.score(reference, abstractive)

In [40]:
def format_scores(scores):
    return {
        "ROUGE-1": scores["rouge1"].fmeasure,
        "ROUGE-2": scores["rouge2"].fmeasure,
        "ROUGE-L": scores["rougeL"].fmeasure,
    }

results = pd.DataFrame.from_dict({
    "Extractive (TextRank)": format_scores(scores_extractive),
    "Abstractive (Transformer)": format_scores(scores_abstractive)
}, orient="columns")

results = results.round(4)
results

,Extractive (TextRank),Abstractive (Transformer)
ROUGE-1,0.1750,0.1695
ROUGE-2,0.0084,0.0256
ROUGE-L,0.0917,0.1102


In [41]:
verbose = []

for name, scores in [("Extractive", scores_extractive), ("Abstractive", scores_abstractive)]:
    for k in ["rouge1", "rouge2", "rougeL"]:
        verbose.append({
            "Model": name,
            "Metric": k.upper(),
            "Precision": scores[k].precision,
            "Recall": scores[k].recall,
            "F1": scores[k].fmeasure
        })

pd.DataFrame(verbose).round(4)

,Model,Metric,Precision,Recall,F1
0,Extractive,ROUGE1,0.4565,0.1082,0.1750
1,Extractive,ROUGE2,0.0222,0.0052,0.0084
2,Extractive,ROUGEL,0.2391,0.0567,0.0917
3,Abstractive,ROUGE1,0.4762,0.1031,0.1695
4,Abstractive,ROUGE2,0.0732,0.0155,0.0256
5,Abstractive,ROUGEL,0.3095,0.0670,0.1102
